### 03  OHLCV Data Quality
Cel: ocena kompletności, poprawności i przydatności danych rynkowych (ceny, wolumen, returny).
Tabele: silver.ohlcv_indicators, gold.ohlcv_with_dimension, gold.sentiment_vs_returns, gold.sentiment_lead_lag

In [0]:
%sql
DESCRIBE silver.ohlcv_indicators

In [0]:
%sql
SELECT symbol,
  COUNT(*) AS total_rows,
  MIN(date) AS min_date,
  MAX(date) AS max_date,
  COUNT (*) - COUNT(close) AS null_close
FROM silver.ohlcv_indicators
GROUP BY symbol
ORDER BY total_rows ASC

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN close <= 0 THEN 1 ELSE 0 END) AS zero_or_negative,
  MIN(close) AS min_close,
  MAX(close) AS max_close
FROM silver.ohlcv_indicators

In [0]:
%sql
SELECT
  SUM(CASE WHEN volume < 0 THEN 1 ELSE 0 END) AS negative_volume,
  SUM(CASE WHEN volume = 0 THEN 1 ELSE 0 END) AS zero_volume,
  COUNT(*) AS total_rows
FROM silver.ohlcv_indicators

In [0]:
%sql
SELECT symbol, 
  COUNT(*) AS zero_volume_days
FROM silver.ohlcv_indicators
WHERE volume = 0
GROUP BY symbol
ORDER BY zero_volume_days DESC

In [0]:
%sql
SELECT
  COUNT(DISTINCT symbol) AS total_symbols,
  SUM(CASE WHEN sector IS NULL THEN 1 ELSE 0 END) AS missing_sector,
  COUNT(DISTINCT sector) AS sector_count
FROM gold.ohlcv_with_dimension

In [0]:
%sql
SELECT DISTINCT symbol
FROM gold.ohlcv_with_dimension
WHERE sector IS NULL

In [0]:
%sql
SELECT
  sector,
  COUNT(DISTINCT symbol) AS symbol_count,
  COUNT(*) AS total_rows
FROM gold.ohlcv_with_dimension
WHERE sector IS NOT NULL
GROUP BY sector
ORDER BY symbol_count DESC

1. Wszystkie symbole mają zero nulli w close. SOLS - dane od 2025-10-20 (110 rekordów vs 265 u reszty), prawdopodobnie późniejsze dodanie do QQQ.
2. Zero ujemnych lub zerowych cen close. Zakres 7.69-5815.92 - realistyczny.
3. 265 zerowych wolumenów - wszystkie w ^VIX (indeks, nie jest handlowany). Brak ujemnych wolumenów. Dane wolumenowe czyste.
4. Ciągłość szeregów czasowych ok - jedyny outlier to SOLS, reszta symboli ma identyczną liczbę rekordów (265).
5. 6 symboli bez sektora (^VIX, QQQ, SPY, TLT, GLD, USD) - benchmarki, nie spółki QQQ. Expected. Wszystkie spółki QQQ mają pełne pokrycie sektorowe.
6. Technology dominuje z 41 symbolami, zgodnie z profilem QQQ. Sektory z 1-2 symbolami (financial services, real estate, energy, basic materials) - zbyt mała próbka na agregacje sektorowe. Dashboard powinien filtrować lub oznaczać sektory z <5 symbolami.